# 03 — Pré-processamento das Imagens (PDI)

> **Etapa 2 do refactor.** Este notebook trata **apenas** as imagens. Os metadados
> já foram processados no `02_preprocessamento_metadados.ipynb`. A junção das
> duas fontes acontece no notebook de extração de features.

## O que este notebook faz

1. Carrega os splits do nb02 (`train/val/test.parquet`) só para saber **quais
   imagens processar** e preservar o mesmo particionamento.
2. Aplica um pipeline de 4 etapas em cada imagem:
   1. **Hair removal** (DullRazor — blackhat + inpaint)
   2. **Color constancy** (Shades of Gray)
   3. **Segmentação da lesão** (Otsu no canal L* do LAB + fallback)
   4. **Crop quadrado da bbox + resize 256×256**
3. Persiste cada imagem processada como `.npz` contendo `{image, mask}`.
4. Registra métricas de qualidade (quantas caíram em fallback de segmentação).

## O que este notebook NÃO faz

- Não extrai features (isso é o próximo notebook).
- Não treina modelo.
- Não toca em metadados (já resolvido no nb02).

## Saída final

```
data/processed/images/
├── PAT_1516_1765_530.npz   # {image: uint8 (256,256,3), mask: uint8 (256,256)}
├── PAT_46_881_939.npz
├── ...
└── _manifest.json          # lista de img_id processados + flag de fallback
```


## 1. Setup

In [ ]:
from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from tqdm.auto import tqdm

ROOT        = Path('..').resolve()
RAW_IMG     = ROOT / 'data' / 'raw' / 'images'
META_DIR    = ROOT / 'data' / 'processed' / 'metadata'
OUT_DIR     = ROOT / 'data' / 'processed' / 'images'
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SIZE = 256
PADDING     = 0.10          # 10% de padding em volta da bbox da lesão

print('input :', RAW_IMG)
print('output:', OUT_DIR)
print('target:', f'{TARGET_SIZE}×{TARGET_SIZE}')


## 2. Carregamento dos splits do nb02

Não vamos refazer o split — carregamos os três parquets gerados no notebook 02
só para saber **quais `img_id` processar** e manter rastreabilidade por split.


In [ ]:
train_df = pd.read_parquet(META_DIR / 'train.parquet')[['img_id', 'diagnostic']]
val_df   = pd.read_parquet(META_DIR / 'val.parquet')  [['img_id', 'diagnostic']]
test_df  = pd.read_parquet(META_DIR / 'test.parquet') [['img_id', 'diagnostic']]

all_df = pd.concat([
    train_df.assign(split='train'),
    val_df.assign(split='val'),
    test_df.assign(split='test'),
], ignore_index=True)

print(f'train : {len(train_df)}')
print(f'val   : {len(val_df)}')
print(f'test  : {len(test_df)}')
print(f'total : {len(all_df)}')

# Integridade: todo img_id aponta para um arquivo existente
missing = [i for i in all_df.img_id if not (RAW_IMG / i).exists()]
assert not missing, f'{len(missing)} imagens não encontradas no disco'
print(f'✅ Todas as {len(all_df)} imagens existem em {RAW_IMG}')


## 3. Decisões do pipeline

Quatro etapas, cada uma resolve um problema específico das fotos clínicas do
PAD-UFES. A ordem importa: color constancy **antes** da segmentação (Otsu
fica mais estável com iluminação normalizada), hair removal **antes** do
color constancy (o pelo distorce a estimativa de iluminante).

| # | Etapa | Resolve |
|---|---|---|
| 1 | **Hair removal** (DullRazor) | Pelo atravessando lesão inflaciona features de textura (GLCM, LBP, Gabor) |
| 2 | **Color constancy** (Shades of Gray) | Smartphones diferentes têm balanço de branco diferente → features de cor ruidosas |
| 3 | **Segmentação** (Otsu no L* do LAB, invertido) | Sem segmentar, features são extraídas sobre lesão + pele normal + fundo misturados |
| 4 | **Crop bbox quadrada + resize 256** | Preserva aspect ratio (18% das imagens não são quadradas — resize direto distorce) |

### Fallback de segmentação

Otsu falha em ~5-10% dos casos em datasets dermatológicos (lesão muito clara,
flash, pele muito danificada). Quando a máscara resultante tiver área < 5%
ou > 95% da imagem, usamos um **center-crop retangular (60% da imagem)** como
fallback. A flag `fallback=True` fica registrada no manifesto para auditoria.


## 4. Funções do pipeline

In [ ]:
def hair_removal(img_rgb: np.ndarray) -> np.ndarray:
    """DullRazor: detecta pelo via blackhat morfológico e reconstrói via inpaint."""
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (17, 17))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    _, hair_mask = cv2.threshold(blackhat, 10, 255, cv2.THRESH_BINARY)
    return cv2.inpaint(img_rgb, hair_mask, 3, cv2.INPAINT_TELEA)


In [ ]:
def shades_of_gray(img_rgb: np.ndarray, p: int = 6) -> np.ndarray:
    """
    Color constancy de Finlayson: estima o iluminante como a norma-p da imagem
    por canal e divide cada canal por esse valor, normalizando o brilho global.
    """
    img = img_rgb.astype(np.float32)
    means = np.power(np.mean(np.power(img, p), axis=(0, 1)), 1.0 / p)
    # Normaliza o vetor de iluminante para preservar magnitude global
    means = means / np.sqrt(np.sum(means ** 2)) * np.sqrt(3)
    corrected = img / means[np.newaxis, np.newaxis, :]
    return np.clip(corrected, 0, 255).astype(np.uint8)


In [ ]:
def _center_mask(shape) -> np.ndarray:
    """Fallback: máscara retangular central ocupando 60% × 60% da imagem."""
    h, w = shape
    mask = np.zeros((h, w), dtype=np.uint8)
    pad_h, pad_w = int(h * 0.2), int(w * 0.2)
    mask[pad_h:h - pad_h, pad_w:w - pad_w] = 255
    return mask


def segment_lesion(img_rgb: np.ndarray) -> tuple[np.ndarray, bool]:
    """
    Segmenta a lesão via Otsu no canal L* do LAB (invertido).
    Retorna (máscara uint8, fallback_usado).
    """
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    L_inv = 255 - lab[:, :, 0]   # lesão tende a ser mais escura que pele

    _, mask = cv2.threshold(L_inv, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Morfologia: remove ruído + preenche buracos
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)

    # Maior componente conectado (remove segmentações espúrias)
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if n_labels <= 1:
        return _center_mask(img_rgb.shape[:2]), True

    largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    mask = (labels == largest).astype(np.uint8) * 255

    # Plausibilidade: entre 5% e 95% da imagem
    area_ratio = float((mask > 0).mean())
    if area_ratio < 0.05 or area_ratio > 0.95:
        return _center_mask(img_rgb.shape[:2]), True

    return mask, False


In [ ]:
def crop_square_bbox(img_rgb: np.ndarray, mask: np.ndarray,
                     padding: float = 0.10) -> tuple[np.ndarray, np.ndarray]:
    """Recorta bbox quadrada centrada na máscara, com padding relativo."""
    ys, xs = np.where(mask > 0)
    if len(ys) == 0:
        return img_rgb, mask

    y0, y1 = int(ys.min()), int(ys.max())
    x0, x1 = int(xs.min()), int(xs.max())
    cy, cx = (y0 + y1) / 2, (x0 + x1) / 2
    side   = max(y1 - y0, x1 - x0)
    side   = int(side * (1 + 2 * padding))
    half   = side // 2

    h, w = img_rgb.shape[:2]
    y_start = max(0, int(cy - half)); y_end = min(h, int(cy + half))
    x_start = max(0, int(cx - half)); x_end = min(w, int(cx + half))

    return img_rgb[y_start:y_end, x_start:x_end], mask[y_start:y_end, x_start:x_end]


In [ ]:
def process_image(img_path: Path, target_size: int = TARGET_SIZE
                  ) -> tuple[np.ndarray, np.ndarray, bool]:
    """
    Pipeline completo para uma imagem:
        load → hair removal → color constancy → segmentação → crop → resize
    Retorna (image uint8 HxWx3, mask uint8 HxW, fallback_usado).
    """
    img = np.array(Image.open(img_path).convert('RGB'))
    img = hair_removal(img)
    img = shades_of_gray(img, p=6)
    mask, fallback = segment_lesion(img)
    img_crop, mask_crop = crop_square_bbox(img, mask, padding=PADDING)

    img_out  = cv2.resize(img_crop,  (target_size, target_size), interpolation=cv2.INTER_LANCZOS4)
    mask_out = cv2.resize(mask_crop, (target_size, target_size), interpolation=cv2.INTER_NEAREST)
    return img_out, mask_out, fallback


## 5. Sanity visual — 2 amostras por classe

Antes de rodar nas 2298 imagens, validamos visualmente que o pipeline funciona
em casos representativos. Se a segmentação visivelmente falhar em melanomas
(a classe mais crítica), paramos aqui e ajustamos Otsu antes do bulk.


In [ ]:
CLASS_ORDER = ['BCC', 'ACK', 'NEV', 'SEK', 'SCC', 'MEL']
SAMPLES_PER_CLASS = 2

# Pegar amostras fixas (seed) do train de cada classe
rng = np.random.default_rng(42)
sanity_ids = []
for cls in CLASS_ORDER:
    pool = train_df.loc[train_df.diagnostic == cls, 'img_id'].tolist()
    picked = rng.choice(pool, size=SAMPLES_PER_CLASS, replace=False)
    sanity_ids.extend([(cls, i) for i in picked])

fig, axes = plt.subplots(len(sanity_ids), 5, figsize=(13, 2.4 * len(sanity_ids)))
col_titles = ['Original', '+ hair removal', '+ color constancy', 'Máscara', 'Final 256×256']

for row, (cls, img_id) in enumerate(sanity_ids):
    path = RAW_IMG / img_id
    orig = np.array(Image.open(path).convert('RGB'))
    nohair = hair_removal(orig)
    norm   = shades_of_gray(nohair)
    mask, fb = segment_lesion(norm)
    crop_img, crop_mask = crop_square_bbox(norm, mask, padding=PADDING)
    final = cv2.resize(crop_img, (TARGET_SIZE, TARGET_SIZE), interpolation=cv2.INTER_LANCZOS4)

    # overlay de máscara na imagem normalizada
    overlay = norm.copy()
    overlay[mask == 0] = (overlay[mask == 0] * 0.3).astype(np.uint8)

    for ax, arr in zip(axes[row], [orig, nohair, norm, overlay, final]):
        ax.imshow(arr); ax.set_xticks([]); ax.set_yticks([])

    label = f'{cls}' + (' [FB]' if fb else '')
    axes[row, 0].set_ylabel(label, fontsize=10, rotation=0, ha='right', va='center', labelpad=30)

for ax, t in zip(axes[0], col_titles):
    ax.set_title(t, fontsize=10)

plt.tight_layout(); plt.show()
print('→ [FB] marca casos onde a segmentação Otsu caiu no fallback de center-crop.')


## 6. Execução em batch

Processamos todas as 2298 imagens (train + val + test) e salvamos cada uma
como `.npz`. Se o arquivo já existir, pulamos — isso torna o notebook idempotente
(dá para re-rodar só para as imagens que faltaram, sem re-processar tudo).


In [ ]:
manifest = {}
errors   = []
t0 = time.time()

for _, row in tqdm(all_df.iterrows(), total=len(all_df), desc='Processando'):
    img_id = row['img_id']
    out_path = OUT_DIR / f'{img_id[:-4]}.npz'   # remove .png, adiciona .npz

    if out_path.exists():
        # Já processado — ainda precisa entrar no manifest
        data = np.load(out_path)
        manifest[img_id] = {
            'file': out_path.name,
            'split': row['split'],
            'diagnostic': row['diagnostic'],
            'fallback': bool(data.get('fallback', False)),
        }
        continue

    try:
        img_out, mask_out, fallback = process_image(RAW_IMG / img_id)
        np.savez_compressed(out_path,
                            image=img_out,
                            mask=mask_out,
                            fallback=np.array(fallback))
        manifest[img_id] = {
            'file': out_path.name,
            'split': row['split'],
            'diagnostic': row['diagnostic'],
            'fallback': bool(fallback),
        }
    except Exception as e:
        errors.append((img_id, str(e)))

elapsed = time.time() - t0
print(f'\nConcluído em {elapsed:.1f}s  ({elapsed/len(all_df)*1000:.0f} ms/img)')
print(f'Processadas : {len(manifest)}')
print(f'Falhas      : {len(errors)}')
if errors:
    for i, e in errors[:5]:
        print(f'  {i}: {e}')

# Salva manifesto
with open(OUT_DIR / '_manifest.json', 'w') as f:
    json.dump({
        'target_size': TARGET_SIZE,
        'padding': PADDING,
        'total': len(manifest),
        'errors': len(errors),
        'images': manifest,
    }, f, indent=1)
print(f'✅ Manifesto salvo: {OUT_DIR / "_manifest.json"}')


## 7. Métricas de qualidade

Duas perguntas para auditoria:
1. **Quantas imagens caíram no fallback de segmentação?** (Otsu falhou)
2. **O fallback está concentrado em alguma classe?** (se MEL tiver muito fallback, é um problema)


In [ ]:
mdf = pd.DataFrame.from_dict(manifest, orient='index')

total_fb = int(mdf.fallback.sum())
pct_fb   = total_fb / len(mdf) * 100
print(f'Fallbacks totais: {total_fb} / {len(mdf)}  ({pct_fb:.1f}%)')
print()

print('Fallbacks por classe:')
by_class = mdf.groupby('diagnostic').agg(
    n=('fallback', 'size'),
    fb=('fallback', 'sum'),
)
by_class['fb_%'] = (by_class['fb'] / by_class['n'] * 100).round(1)
print(by_class.reindex(CLASS_ORDER).to_string())

print()
print('Fallbacks por split:')
by_split = mdf.groupby('split').agg(
    n=('fallback', 'size'),
    fb=('fallback', 'sum'),
)
by_split['fb_%'] = (by_split['fb'] / by_split['n'] * 100).round(1)
print(by_split.to_string())


## 8. Função pública `load_processed`

Helper para o próximo notebook. Carrega uma imagem processada por `img_id`
sem precisar saber onde está ou em que formato.


In [ ]:
def load_processed(img_id: str, base_dir: Path = OUT_DIR
                   ) -> tuple[np.ndarray, np.ndarray]:
    '''Carrega (image, mask) processadas a partir do img_id do metadata.'''
    stem = img_id[:-4] if img_id.endswith('.png') else img_id
    data = np.load(base_dir / f'{stem}.npz')
    return data['image'], data['mask']


# Teste: carrega a primeira imagem do train e mostra image + mask lado a lado
sample_id = train_df.img_id.iloc[0]
img, mask = load_processed(sample_id)
print(f'sample: {sample_id}')
print(f'  image shape: {img.shape}  dtype: {img.dtype}')
print(f'  mask shape : {mask.shape}  dtype: {mask.dtype}')
print(f'  mask area  : {(mask>0).mean()*100:.1f}% da imagem')

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(img);                    axes[0].set_title('image');  axes[0].axis('off')
axes[1].imshow(mask, cmap='gray');      axes[1].set_title('mask');   axes[1].axis('off')
overlay = img.copy()
overlay[mask == 0] = (overlay[mask == 0] * 0.3).astype(np.uint8)
axes[2].imshow(overlay);                axes[2].set_title('overlay'); axes[2].axis('off')
plt.tight_layout(); plt.show()


## 9. Resumo

```
┌─ INPUT ────────────────────────────────────────────┐
│  data/raw/images/*.png                             │
│  2298 imagens clínicas de smartphone               │
│  189–3096 px, RGB/RGBA, com pelo/flash/sombra      │
└────────────────────────────────────────────────────┘
                        ↓
┌─ PIPELINE (por imagem) ────────────────────────────┐
│  1. Hair removal      (DullRazor)                  │
│  2. Color constancy   (Shades of Gray, p=6)        │
│  3. Segmentação       (Otsu LAB L* + fallback)     │
│  4. Crop bbox quadrada + resize 256×256            │
└────────────────────────────────────────────────────┘
                        ↓
┌─ OUTPUT ───────────────────────────────────────────┐
│  data/processed/images/                            │
│    {img_id}.npz  (image 256×256×3, mask 256×256)   │
│    _manifest.json (fallbacks, classes, splits)     │
│                                                    │
│  load_processed(img_id) → (image, mask)            │
└────────────────────────────────────────────────────┘
```

### Próximo notebook

`04_extracao_features_pdi.ipynb` — extrai features clássicas (cor, textura
GLCM/LBP, Gabor, forma) **apenas dentro da máscara** de cada imagem,
combina com as features tabulares do nb02 via `img_id`, e salva a matriz
final pronta para modelagem.
